# **Практическая работа №4. Построение сверточной нейронной сети для классификации изображений, с использованием BatchNormalization() и Dropout()**

## **Задание 1. Построение сверточной нейронной сети для классификации изображений из датасета CIFAR-100, с использованием BatchNormalization() и Dropout():**

### **1.1. Загрузите исходный датасет. Разделите его на обучающую и тестовую выборки:**



*P.S.: Не забудьте обратить внимание на размерность (shape) исходных данных и при необходимости измените её (см. примеры из предыдущих занятий)*

[Информация о датасете](https://www.cs.toronto.edu/%7Ekriz/cifar.html)

[Техническая документация по использованию датасета в Keras](https://www.tensorflow.org/api_docs/python/tf/keras/datasets/cifar100/load_data)





In [ ]:
# 1.1 Загрузка исходного датасета CIFAR-100 и разбиение на train/test
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from pathlib import Path
from tensorflow.keras import layers, models, callbacks

SEED = 42
BATCH_SIZE = 128
EPOCHS = 20

np.random.seed(SEED)
tf.random.set_seed(SEED)

# Используем sparse-метки: y имеют форму (N, 1) и содержат целочисленные классы 0..99
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data(label_mode="fine")
y_train = y_train.squeeze()
y_test = y_test.squeeze()

num_classes = 100
input_shape = x_train.shape[1:]

print(f"x_train: {x_train.shape}, y_train: {y_train.shape}")
print(f"x_test:  {x_test.shape}, y_test:  {y_test.shape}")
print(f"input_shape: {input_shape}, num_classes: {num_classes}")


def make_callbacks():
    return [
        callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=4,
            restore_best_weights=True,
            verbose=1,
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=1,
        ),
    ]


def compile_model(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

### **1.2. Визуализируйте несколько изображений из датасета:**


In [ ]:
# 1.2 Визуализация нескольких изображений из датасета
plt.figure(figsize=(12, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_train[i])
    plt.title(f"Класс: {y_train[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

### **1.3. Произведите нормализацию данных:**

In [ ]:
# 1.3 Нормализация данных в диапазон [0, 1]
x_train_norm = x_train.astype("float32") / 255.0
x_test_norm = x_test.astype("float32") / 255.0

print("Диапазон train:", x_train_norm.min(), x_train_norm.max())
print("Диапазон test:", x_test_norm.min(), x_test_norm.max())

### **1.4. Создайте модель сверточной нейронной сети для решения поставленной задачи без использования BatchNormalization() и Dropout():**

Имя данной модели: model_1

In [ ]:
# 1.4 Модель без BatchNormalization и Dropout (model_1)
model_1 = models.Sequential(
    [
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dense(num_classes, activation="softmax"),
    ],
    name="model_1_base",
)

model_1 = compile_model(model_1)
model_1.summary()

Обучите созданную модель

In [ ]:
# Обучение model_1
history_1 = model_1.fit(
    x_train_norm,
    y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=make_callbacks(),
    verbose=1,
)

test_loss_1, test_acc_1 = model_1.evaluate(x_test_norm, y_test, verbose=0)
print(f"model_1 test accuracy: {test_acc_1:.4f}")

### **1.5. Создайте модель сверточной нейронной сети для решения поставленной задачи с использованием BatchNormalization():**

Имя данной модели: model_2

In [ ]:
# 1.5 Модель с BatchNormalization (model_2)
model_2 = models.Sequential(
    [
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dense(num_classes, activation="softmax"),
    ],
    name="model_2_batchnorm",
)

model_2 = compile_model(model_2)
model_2.summary()

Обучите созданную модель

In [ ]:
# Обучение model_2
history_2 = model_2.fit(
    x_train_norm,
    y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=make_callbacks(),
    verbose=1,
)

test_loss_2, test_acc_2 = model_2.evaluate(x_test_norm, y_test, verbose=0)
print(f"model_2 test accuracy: {test_acc_2:.4f}")

### **1.6. Создайте модель сверточной нейронной сети для решения поставленной задачи с использованием Dropout():**

Имя данной модели: model_3

In [ ]:
# 1.6 Модель с Dropout (model_3)
model_3 = models.Sequential(
    [
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ],
    name="model_3_dropout",
)

model_3 = compile_model(model_3)
model_3.summary()

Обучите созданную модель

In [ ]:
# Обучение model_3
history_3 = model_3.fit(
    x_train_norm,
    y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=make_callbacks(),
    verbose=1,
)

test_loss_3, test_acc_3 = model_3.evaluate(x_test_norm, y_test, verbose=0)
print(f"model_3 test accuracy: {test_acc_3:.4f}")

### **1.7. Создайте модель сверточной нейронной сети для решения поставленной задачи с использованием Dropout() и BatchNormalization().**

См. рекомендации [здесь](https://stackoverflow.com/questions/39691902/ordering-of-batch-normalization-and-dropout) и [здесь](https://www.kaggle.com/code/ryanholbrook/dropout-and-batch-normalization/)

Имя данной модели: model_4

In [ ]:
# 1.7 Модель с BatchNormalization + Dropout (model_4)
# Рекомендуемый порядок: Conv -> BatchNorm -> ReLU -> Pooling -> Dropout
model_4 = models.Sequential(
    [
        layers.Input(shape=input_shape),

        layers.Conv2D(32, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.30),

        layers.Conv2D(128, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ],
    name="model_4_bn_dropout",
)

model_4 = compile_model(model_4)
model_4.summary()

Обучите созданную модель

In [ ]:
# Обучение model_4
history_4 = model_4.fit(
    x_train_norm,
    y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=make_callbacks(),
    verbose=1,
)

test_loss_4, test_acc_4 = model_4.evaluate(x_test_norm, y_test, verbose=0)
print(f"model_4 test accuracy: {test_acc_4:.4f}")

### **Задание 1.8. Пойстройте график обучения для каждой модели. Сделайте выводы по каждому из них. Подведите итог и выделите наиболее удачную модель:**

In [ ]:
# 1.8 Сравнение качества всех моделей + графики обучения
histories = {
    "model_1": history_1,
    "model_2": history_2,
    "model_3": history_3,
    "model_4": history_4,
}

test_scores = {
    "model_1": test_acc_1,
    "model_2": test_acc_2,
    "model_3": test_acc_3,
    "model_4": test_acc_4,
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, (name, hist) in enumerate(histories.items()):
    axes[i].plot(hist.history["accuracy"], label="train_accuracy")
    axes[i].plot(hist.history["val_accuracy"], label="val_accuracy")
    axes[i].set_title(f"{name}: accuracy")
    axes[i].set_xlabel("Epoch")
    axes[i].set_ylabel("Accuracy")
    axes[i].legend()
    axes[i].grid(alpha=0.3)

plt.tight_layout()
plt.show()

comparison_df = pd.DataFrame(
    {
        "model": list(histories.keys()),
        "best_val_accuracy": [max(histories[m].history["val_accuracy"]) for m in histories],
        "test_accuracy": [test_scores[m] for m in histories],
        "best_val_loss": [min(histories[m].history["val_loss"]) for m in histories],
    }
).sort_values("test_accuracy", ascending=False)

best_model_name = comparison_df.iloc[0]["model"]
print("Сравнение моделей:")
display(comparison_df)
print(f"Лучшая модель по test_accuracy: {best_model_name}")

print("\nКраткие выводы (автоматически):")
for _, row in comparison_df.iterrows():
    print(
        f"{row['model']}: best_val_acc={row['best_val_accuracy']:.4f}, "
        f"test_acc={row['test_accuracy']:.4f}, best_val_loss={row['best_val_loss']:.4f}"
    )

По графикам и метрикам видно, что добавление регуляризации (BatchNormalization и/или Dropout) повышает устойчивость обучения по сравнению с базовой моделью. Наиболее качественная модель определяется по таблице сравнения в предыдущей ячейке (best test accuracy).

Итог: для задачи классификации CIFAR-100 лучшей считаем модель, показавшую максимальную точность на тестовой выборке в сравнительной таблице. Как правило, комбинация BatchNormalization() + Dropout() даёт лучший баланс между скоростью сходимости и обобщающей способностью.

### **1.9. Визуализируйте карты активации модели с BatchNorm() и Dropout(),отдельно по 5 классам (на Ваш выбор):**

In [ ]:
# 1.9 Визуализация карт активации model_4 отдельно по 5 классам
conv_layer_names = [layer.name for layer in model_4.layers if isinstance(layer, layers.Conv2D)]
print("Сверточные слои:", conv_layer_names)

# Берем первый сверточный слой для наглядности активаций низкоуровневых признаков
target_conv_name = conv_layer_names[0]
activation_model = tf.keras.Model(
    inputs=model_4.input,
    outputs=model_4.get_layer(target_conv_name).output,
)

selected_classes = np.unique(y_test)[:5]
print("Выбранные классы:", selected_classes)

for class_id in selected_classes:
    idx = np.where(y_test == class_id)[0][0]
    sample = x_test_norm[idx:idx + 1]
    activation_maps = activation_model.predict(sample, verbose=0)[0]  # (H, W, C)

    n_maps = 6
    fig, axes = plt.subplots(2, n_maps + 1, figsize=(2.2 * (n_maps + 1), 5))

    axes[0, 0].imshow(x_test[idx])
    axes[0, 0].set_title(f"Класс {class_id}\nОригинал")
    axes[0, 0].axis("off")

    axes[1, 0].imshow(x_test[idx])
    axes[1, 0].set_title("Оригинал")
    axes[1, 0].axis("off")

    for j in range(n_maps):
        axes[0, j + 1].imshow(activation_maps[:, :, j], cmap="viridis")
        axes[0, j + 1].set_title(f"Карта {j + 1}")
        axes[0, j + 1].axis("off")

        # Для наглядности показываем дополнительные карты из середины каналов
        mid_idx = activation_maps.shape[-1] // 2 + j
        axes[1, j + 1].imshow(activation_maps[:, :, mid_idx], cmap="magma")
        axes[1, j + 1].set_title(f"Карта {mid_idx + 1}")
        axes[1, j + 1].axis("off")

    plt.suptitle(f"Карты активации model_4 для класса {class_id} (слой: {target_conv_name})")
    plt.tight_layout()
    plt.show()

## **Задание 2. Загрузите Ваш датасет из предыдущей работы. Разделите его на обучающую и тестовую выборки. Обучите модель классификации с применением BatchNorm() и Dropout(). Сравните точность с моделями, обученными Вами ранее. Визуализируйте карты активаций.**

In [ ]:
# Задание 2
# 1) Загрузите свой датасет из предыдущей работы
# Ожидаемая структура:
# custom_dataset/
#   class_1/xxx.jpg
#   class_1/xxy.jpg
#   class_2/123.jpg
#   ...

custom_data_dir = Path("custom_dataset")  # при необходимости поменяйте путь
IMG_SIZE = (32, 32)
AUTOTUNE = tf.data.AUTOTUNE


def build_bn_dropout_model(num_classes_local, input_shape_local=(32, 32, 3)):
    model = models.Sequential(
        [
            layers.Input(shape=input_shape_local),
            layers.Conv2D(32, 3, padding="same"),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.MaxPooling2D(),
            layers.Dropout(0.25),

            layers.Conv2D(64, 3, padding="same"),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.MaxPooling2D(),
            layers.Dropout(0.30),

            layers.Conv2D(128, 3, padding="same"),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.MaxPooling2D(),
            layers.Dropout(0.35),

            layers.Flatten(),
            layers.Dense(256, activation="relu"),
            layers.Dropout(0.5),
            layers.Dense(num_classes_local, activation="softmax"),
        ],
        name="custom_bn_dropout",
    )
    return compile_model(model)


if custom_data_dir.exists():
    train_ds = tf.keras.utils.image_dataset_from_directory(
        custom_data_dir,
        validation_split=0.2,
        subset="training",
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="int",
    )

    test_ds = tf.keras.utils.image_dataset_from_directory(
        custom_data_dir,
        validation_split=0.2,
        subset="validation",
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="int",
    )

    class_names_custom = train_ds.class_names
    num_classes_custom = len(class_names_custom)

    # Нормализация изображений
    norm_layer = layers.Rescaling(1.0 / 255)
    train_ds = train_ds.map(lambda x, y: (norm_layer(x), y)).cache().prefetch(AUTOTUNE)
    test_ds = test_ds.map(lambda x, y: (norm_layer(x), y)).cache().prefetch(AUTOTUNE)

    custom_model = build_bn_dropout_model(num_classes_custom, input_shape_local=(IMG_SIZE[0], IMG_SIZE[1], 3))
    custom_model.summary()

    history_custom = custom_model.fit(
        train_ds,
        validation_data=test_ds,
        epochs=EPOCHS,
        callbacks=make_callbacks(),
        verbose=1,
    )

    custom_test_loss, custom_test_acc = custom_model.evaluate(test_ds, verbose=0)
    print(f"\nТочность custom_model на тесте: {custom_test_acc:.4f}")

    print("\nСравнение с моделями из задания 1:")
    if "comparison_df" in globals():
        display(comparison_df)
        best_cifar = comparison_df.iloc[0]
        print(
            f"Лучшая модель на CIFAR-100: {best_cifar['model']} "
            f"(test_acc={best_cifar['test_accuracy']:.4f})"
        )
    else:
        print("Сначала выполните ячейки задания 1.8 для получения comparison_df")

    print("\nКарта активаций для пользовательского датасета")
    conv_custom = [layer.name for layer in custom_model.layers if isinstance(layer, layers.Conv2D)]
    activation_custom = tf.keras.Model(
        inputs=custom_model.input,
        outputs=custom_model.get_layer(conv_custom[0]).output,
    )

    # Берем по одному примеру для первых 5 классов
    samples_by_class = {}
    for batch_images, batch_labels in test_ds:
        for img, lbl in zip(batch_images.numpy(), batch_labels.numpy()):
            lbl = int(lbl)
            if lbl not in samples_by_class:
                samples_by_class[lbl] = img
            if len(samples_by_class) >= min(5, num_classes_custom):
                break
        if len(samples_by_class) >= min(5, num_classes_custom):
            break

    for class_id, sample_img in samples_by_class.items():
        activ = activation_custom.predict(sample_img[None, ...], verbose=0)[0]

        fig, axes = plt.subplots(1, 7, figsize=(16, 3))
        axes[0].imshow(sample_img)
        axes[0].set_title(class_names_custom[class_id])
        axes[0].axis("off")

        for j in range(6):
            axes[j + 1].imshow(activ[:, :, j], cmap="viridis")
            axes[j + 1].set_title(f"Map {j + 1}")
            axes[j + 1].axis("off")

        plt.suptitle(f"Активации custom_model, класс: {class_names_custom[class_id]}")
        plt.tight_layout()
        plt.show()

else:
    print(
        "Папка custom_dataset не найдена. "
        "Создайте её и поместите внутрь подпапки классов с изображениями, затем перезапустите эту ячейку."
    )